In [53]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

illegal_summary = pd.read_csv('../resources/illegal_summary.csv')
parking_summary = pd.read_csv('../resources/parking_summary.csv')


In [54]:
illegal_summary

,지역,단속건수,면적,면적 당 단속건수
0,새롬동,7922,1.49,5317
1,다정동,8908,1.70,5240
2,아름동,10572,2.19,4827
3,어진동,11016,2.65,4157
4,고운동,15164,5.35,2834
5,대평동,4065,1.50,2710
6,소담동,3086,1.17,2638
7,보람동,3352,1.33,2520
8,도담동,4480,2.03,2207
9,조치원읍,28321,13.56,2089


In [55]:
parking_summary

,지역,주차장수,주차구획수
0,조치원읍,38,1299
1,나성동,4,882
2,보람동,4,238
3,아름동,1,236
4,종촌동,1,160
5,금남면,2,96
6,부강면,5,72
7,전의면,1,65
8,소담동,1,51
9,한솔동,1,33


In [56]:
result = pd.merge(
    illegal_summary,
    parking_summary,
    on='지역',
    how='left'
)
result

,지역,단속건수,면적,면적 당 단속건수,주차장수,주차구획수
0,새롬동,7922,1.49,5317,0,0
1,다정동,8908,1.70,5240,0,0
2,아름동,10572,2.19,4827,1,236
3,어진동,11016,2.65,4157,0,0
4,고운동,15164,5.35,2834,0,0
5,대평동,4065,1.50,2710,0,0
6,소담동,3086,1.17,2638,1,51
7,보람동,3352,1.33,2520,4,238
8,도담동,4480,2.03,2207,1,13
9,조치원읍,28321,13.56,2089,38,1299


In [57]:
result['주차구획당 단속 건수'] = result['단속건수'] / result['주차구획수']
result

,지역,단속건수,면적,면적 당 단속건수,주차장수,주차구획수,주차구획당 단속 건수
0,새롬동,7922,1.49,5317,0,0,inf
1,다정동,8908,1.70,5240,0,0,inf
2,아름동,10572,2.19,4827,1,236,44.796610
3,어진동,11016,2.65,4157,0,0,inf
4,고운동,15164,5.35,2834,0,0,inf
5,대평동,4065,1.50,2710,0,0,inf
6,소담동,3086,1.17,2638,1,51,60.509804
7,보람동,3352,1.33,2520,4,238,14.084034
8,도담동,4480,2.03,2207,1,13,344.615385
9,조치원읍,28321,13.56,2089,38,1299,21.802156


In [58]:
#주차구획이 없는 행정구역 => 주차장 자체가 필요
no_parking = result[result['주차구획수'] == 0]
no_parking

,지역,단속건수,면적,면적 당 단속건수,주차장수,주차구획수,주차구획당 단속 건수
0,새롬동,7922,1.49,5317,0,0,inf
1,다정동,8908,1.70,5240,0,0,inf
3,어진동,11016,2.65,4157,0,0,inf
4,고운동,15164,5.35,2834,0,0,inf
5,대평동,4065,1.50,2710,0,0,inf
12,집현동,2265,5.58,406,0,0,inf
14,해밀동,2748,8.44,326,0,0,inf
15,세종동,3194,23.46,136,0,0,inf
16,산울동,389,2.86,136,0,0,inf
17,반곡동,1935,15.78,123,0,0,inf


In [59]:
ranked_result = result.copy()

ranked_result['주차구획당 단속 건수'] = ranked_result['주차구획당 단속 건수'].replace(np.inf, np.nan)

ranked_result['단속건수 순위'] = ranked_result['단속건수'].rank(ascending=False, method='min')
ranked_result['면적당 단속건수 순위'] = ranked_result['면적 당 단속건수'].rank(ascending=False, method='min')
ranked_result['주차구획당 단속건수 순위'] = ranked_result['주차구획당 단속 건수'].rank(ascending=False, method='min')

ranked_result['종합순위점수'] = (
    ranked_result['단속건수 순위'] +
    ranked_result['면적당 단속건수 순위'] +
    ranked_result['주차구획당 단속건수 순위']
)

ranked_result = ranked_result.sort_values('종합순위점수')

ranked_df = ranked_result[
    ['지역', '단속건수 순위', '면적당 단속건수 순위', '주차구획당 단속건수 순위', '종합순위점수']
].copy()

ranked_df = ranked_df.sort_values('종합순위점수')
ranked_df[['단속건수 순위', '면적당 단속건수 순위', '주차구획당 단속건수 순위', '종합순위점수']] = \
ranked_df[['단속건수 순위', '면적당 단속건수 순위', '주차구획당 단속건수 순위', '종합순위점수']].astype('Int64')
ranked_df

,지역,단속건수 순위,면적당 단속건수 순위,주차구획당 단속건수 순위,종합순위점수
2,아름동,4,3,5,12
9,조치원읍,1,10,6,17
8,도담동,8,9,1,18
6,소담동,12,7,4,23
7,보람동,10,8,7,25
13,나성동,5,14,9,28
11,한솔동,14,12,3,29
10,종촌동,18,11,10,39
19,연동면,20,20,2,42
22,금남면,19,23,8,50


In [60]:
#주차구획 자체가 없는 지역들
na_regions = ranked_df[ranked_df['종합순위점수'].isna()]
na_regions

,지역,단속건수 순위,면적당 단속건수 순위,주차구획당 단속건수 순위,종합순위점수
0,새롬동,7,1,<NA>,<NA>
1,다정동,6,2,<NA>,<NA>
3,어진동,3,4,<NA>,<NA>
4,고운동,2,5,<NA>,<NA>
5,대평동,9,6,<NA>,<NA>
12,집현동,15,13,<NA>,<NA>
14,해밀동,13,15,<NA>,<NA>
15,세종동,11,16,<NA>,<NA>
16,산울동,23,16,<NA>,<NA>
17,반곡동,16,18,<NA>,<NA>


In [61]:
#주차장 없는 행정구역 중 단속건수는 많아서 의미있는 행정구역
#주차 수요는 높은데 공영주차 인프라가 부족한 지역
na_regions['주차구획당 단속건수를 제외한 최종 순위']=na_regions['단속건수 순위']+ na_regions['면적당 단속건수 순위']
na_regions

/var/folders/80/x9ztm2rs5c33hn_3qt3qb6ww0000gn/T/ipykernel_2529/1485716083.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  na_regions['주차구획당 단속건수를 제외한 최종 순위']=na_regions['단속건수 순위']+ na_regions['면적당 단속건수 순위']


,지역,단속건수 순위,면적당 단속건수 순위,주차구획당 단속건수 순위,종합순위점수,주차구획당 단속건수를 제외한 최종 순위
0,새롬동,7,1,<NA>,<NA>,8
1,다정동,6,2,<NA>,<NA>,8
3,어진동,3,4,<NA>,<NA>,7
4,고운동,2,5,<NA>,<NA>,7
5,대평동,9,6,<NA>,<NA>,15
12,집현동,15,13,<NA>,<NA>,28
14,해밀동,13,15,<NA>,<NA>,28
15,세종동,11,16,<NA>,<NA>,27
16,산울동,23,16,<NA>,<NA>,39
17,반곡동,16,18,<NA>,<NA>,34
